# 01 - Cleaning the HPV vaccine acceptance data

**Input:** `s003.csv`, the raw file published with Hawlader et al., PLOS ONE 2024 (2,151 parents, 131 columns).
**Output:** `hpv_clean.csv`, the one clean file that every other notebook reads.

This notebook applies six groups of cleaning decisions. Each section states the decision first, then the code that applies it.

**Rule for every other notebook:** features are all columns that do **not** start with `target_` or `meta_`.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

# The raw file: project folder layout first, then the same folder (for Google Colab uploads).
RAW = next(p for p in [Path("../data/idea2_hpv_vaccine_bangladesh/s003.csv"), Path("s003.csv")] if p.exists())
OUT = Path("../data/hpv_clean.csv") if Path("../data").exists() else Path("hpv_clean.csv")

df = pd.read_csv(RAW)
print("Raw file:", df.shape[0], "rows x", df.shape[1], "columns")

Raw file: 2151 rows x 131 columns


## Decision 1 - Typos

In the published file, "No" was replaced by "0" inside some words ("0akhali", "Don't K0w"). There are also plain spelling mistakes and stray spaces.
**Decision:** fix all of them. The meaning of no value changes.

In [2]:
text_cols = df.select_dtypes(exclude="number").columns
for c in text_cols:
    df[c] = df[c].str.strip()          # " Divorced" -> "Divorced"

typo_fixes = {
    "Dhaka 0rth CC": "Dhaka North CC",
    "0akhali": "Noakhali",
    "Neither important 0r unimportant": "Neither important nor unimportant",
    "Don't K0w": "Don't Know",
    "Busienss Person": "Business Person",
    "Governemnt Job": "Government Job",
    "Strongly Agree": "Strongly agree",   # same capital letters as "Strongly disagree"
}
df = df.replace(typo_fixes)
print("Typos fixed in", len(typo_fixes), "values")

Typos fixed in 7 values


## Decision 2 - Wrong values

For each column we read the question on the form first, then decided.

| Column | Question on the form | Decision |
|---|---|---|
| `educ_spouse` | "Educational Status of your spouse (In completed years)" | 126 rows hold a calendar year (1990-2016) or 143. Set to missing. Imputation happens later, on the training set only. |
| `education_years` | "Educational Status (In completed years)" | Kept. High in the same 126 rows, but consistent with the income of those parents. |
| `occup_spouse` | "Current occupation of your spouse" | The value "8" is the option "Others" (the only option absent from the file; these rows have a job typed in the "specify" column). Replace "8" with "Others". |
| `income` | "Monthly household income (Tk)" | Kept. Large values are outliers, not proven errors. |
| Daughters' ages | "Age of the daughter" | Kept. 49 parents show no daughter aged 9-15 in the file; noted as a limitation. |

In [3]:
wrong_spouse_edu = df["educ_spouse"] > 25
print("educ_spouse values set to missing:", int(wrong_spouse_edu.sum()))
df["educ_spouse"] = df["educ_spouse"].astype("float64")
df.loc[wrong_spouse_edu, "educ_spouse"] = np.nan

print("occup_spouse '8' -> 'Others':", int((df["occup_spouse"] == "8").sum()))
df["occup_spouse"] = df["occup_spouse"].replace({"8": "Others"})

ages = ["age_d1", "age_d2", "age_d3", "age_d4", "age_d5"]
has_eligible = ((df[ages] >= 9) & (df[ages] <= 15)).any(axis=1)
print("Parents with no daughter aged 9-15 in the file (kept):", int((~has_eligible).sum()))

educ_spouse values set to missing: 126
occup_spouse '8' -> 'Others': 143
Parents with no daughter aged 9-15 in the file (kept): 49


## Decision 3 - Rows to drop

- **3 parents have no answer** in both decision questions. A row without a target cannot be used for training or testing. Dropped.
- **1 duplicate row:** two rows are identical in all 130 columns except the serial number. One copy is dropped, so the same parent cannot be in both the training set and the test set.

In [4]:
n_start = len(df)
no_target = df["vac_accept"].isna() | df["vac_if_gov"].isna()
df = df[~no_target]
print("Dropped, no target:", int(no_target.sum()))

duplicate = df.drop(columns="sl").duplicated(keep="first")
df = df[~duplicate].reset_index(drop=True)
print("Dropped, duplicate:", int(duplicate.sum()))
print("Rows:", n_start, "->", len(df))

Dropped, no target: 3
Dropped, duplicate: 1
Rows: 2151 -> 2147


## Decision 4 - The targets

Two questions on the form give the decision:

- `vac_accept`: "Will you give your daughter the HPV vaccine?"
- `vac_if_gov`: "Will you give your daughter the HPV vaccine if the government provided it for free?"

**`target_3class`** (our main target)

| `vac_accept` | `vac_if_gov` | Class |
|---|---|---|
| yes | anything | Yes |
| Not sure / no | yes | Only if free |
| Not sure / no | Not sure / no | Still not yes |

22 parents said yes to the first question but not to the second. They follow the first question: class "Yes".

**`target_2class`** is the authors' own target (`accept`: yes to either question). We keep it only to repeat the authors' logistic regression on a test set.

In [5]:
first_yes = df["vac_accept"] == "yes"
free_yes = df["vac_if_gov"] == "yes"

df["target_3class"] = np.select([first_yes, free_yes], ["Yes", "Only if free"], default="Still not yes")
df["target_2class"] = np.where(first_yes | free_yes, "Accept", "Not accept")

# Our 2-class target must be the same as the authors' column `accept`.
assert ((df["target_2class"] == "Accept") == (df["accept"] == "Yes")).all()

print("Yes at first, not yes if free (class 'Yes'):", int((first_yes & ~free_yes).sum()))
print()
print(df["target_3class"].value_counts().to_string())
print()
print(df["target_2class"].value_counts().to_string())

Yes at first, not yes if free (class 'Yes'): 22

target_3class
Yes              1509
Only if free      354
Still not yes     284

target_2class
Accept        1863
Not accept     284


## Decision 6 - Missing values

(Applied before the columns are dropped, because it uses some of them.)

| Kind | Columns | Rule |
|---|---|---|
| 1. Question skipped, or box not ticked | the four "where did you hear it" blocks (36 columns), `hpv_cervical`, `hpv_transmit`, the four transmission columns | Fill with "No". A parent who never heard of HPV did not hear of it from friends. |
| 2. Question does not apply | `ins_type`, `ins_location` when the daughter is not studying | Fill with "Not studying" |
| 2. Question does not apply | `age_d1` ... `age_d5` (most parents have one daughter) | Replace by `daughter_age_youngest` and `daughter_age_oldest` |
| 3. Truly missing | everything else | Leave as missing. Imputation is fitted on the training set only, in the modelling notebooks. |

In [6]:
source_cols = (
    ["hpv_hcw", "hpv_training", "hpv_friend", "hpv_relatives", "hpv_school", "hpv_pharmacy", "hpv_advert", "hpv_social", "hpv_mass"]
    + ["vac_hcw", "vac_train", "vac_friend", "vac_relative", "vac_school", "vac_pharmacy", "vac_advert", "vac_social", "vac_mass"]
    + ["cancer_hcw", "cancer_training", "cancer_friend", "cancer_relative", "cancer_school", "cancer_pharmacy", "cancer_advert", "cancer_social", "cancer_mass"]
    + ["canvac_hcw", "canvac_training", "canvac_friends", "canvac_relative", "canvac_school", "canvac_pharmacy", "canvac_advert", "canvac_social", "canvac_mass"]
)
skipped_cols = source_cols + ["hpv_cervical", "hpv_transmit", "hpv_sexual", "hpv_blood", "hpv_pubtoilet", "hpv_mtc"]
print("Kind 1: cells filled with 'No':", int(df[skipped_cols].isna().sum().sum()), "in", len(skipped_cols), "columns")
df[skipped_cols] = df[skipped_cols].fillna("No")

not_studying = df["daughter_study"] == "No"
for c in ["ins_type", "ins_location"]:
    df.loc[not_studying & df[c].isna(), c] = "Not studying"
print("Kind 2: parents whose daughter is not studying:", int(not_studying.sum()))

df["daughter_age_youngest"] = df[ages].min(axis=1)
df["daughter_age_oldest"] = df[ages].max(axis=1)

Kind 1: cells filled with 'No': 66335 in 42 columns
Kind 2: parents whose daughter is not studying: 49


## Decision 5 - Columns to drop

| Group | Columns | Why |
|---|---|---|
| A. Data leakage | `vac_accept`, `vac_if_gov`, `accept` | They are the target |
| B. Exact copies | `div`, `dist`, `rel_imp`, `routine_health2`, `epi2`, `sex` | Identical to `division`, `district`, `religion_imp`, `routine_health`, `epi`, `parent` |
| C. Regroupings | `age_cat`, `income_cat`, `occup2`, `marital2`, `family_type`, `checkup`, `routine_health3`, `vul`, `sev`, `ben`, `bar`, `vul_cat`, `sev_cat`, `ben_cat`, `bar_cat`, `cue` | Each is made from a column we keep |
| D. Constant | `eligible` | 1 in every row |
| E. Identifier | `sl` | A counter; it also reveals the place |
| F. Free text | `other_occup`, `other_occup_spouse` | Hand-typed job names |
| (Decision 6) | `age_d1` ... `age_d5` | Replaced by two columns |

`division` and `district` are kept but renamed `meta_division` and `meta_district`. They are **never features**. `meta_division` forms the groups for the leave-one-group-out test.

In [7]:
drop_cols = (
    ["vac_accept", "vac_if_gov", "accept"]
    + ["div", "dist", "rel_imp", "routine_health2", "epi2", "sex"]
    + ["age_cat", "income_cat", "occup2", "marital2", "family_type", "checkup", "routine_health3",
       "vul", "sev", "ben", "bar", "vul_cat", "sev_cat", "ben_cat", "bar_cat", "cue"]
    + ["eligible", "sl", "other_occup", "other_occup_spouse"]
    + ages
)
df = df.drop(columns=drop_cols)
df = df.rename(columns={"division": "meta_division", "district": "meta_district"})

meta_cols = ["meta_division", "meta_district"]
target_cols = ["target_3class", "target_2class"]
feature_cols = [c for c in df.columns if c not in meta_cols + target_cols]
df = df[meta_cols + feature_cols + target_cols]

print("Dropped", len(drop_cols), "columns")
print("Clean file:", len(meta_cols), "meta +", len(feature_cols), "features +", len(target_cols), "targets =", df.shape[1], "columns")

Dropped 34 columns
Clean file: 2 meta + 97 features + 2 targets = 101 columns


## Save, then check

The checks below must all pass. If one fails, the notebook stops with an error.

In [8]:
df.to_csv(OUT, index=False)
check = pd.read_csv(OUT)

assert check.shape == (2147, 101), check.shape
assert check["target_3class"].value_counts().to_dict() == {"Yes": 1509, "Only if free": 354, "Still not yes": 284}
assert check["target_2class"].value_counts().to_dict() == {"Accept": 1863, "Not accept": 284}
assert not set(check.columns) & {"vac_accept", "vac_if_gov", "accept", "sl", "division", "district"}
assert check.duplicated().sum() == 0
assert check["target_3class"].notna().all() and check["meta_division"].notna().all()
text = check.select_dtypes(exclude="number")
assert not text.apply(lambda s: s.str.contains(r"[A-Za-z]0|0[A-Za-z]|^0$", na=False)).any().any(), "a '0' typo is left"
assert check["educ_spouse"].max() <= 25

print("All checks passed. Saved:", OUT, "|", check.shape[0], "rows x", check.shape[1], "columns")
print()
print("Majority-class baseline (3 classes):", round(100 * (check["target_3class"] == "Yes").mean(), 1), "of 100")
print()
print("Missing values left (to impute on the training set):")
missing = check.isna().sum()
print(missing[missing > 0].to_string())
print()
print("Parents per division:")
print(check["meta_division"].value_counts().to_string())

All checks passed. Saved: ../data/hpv_clean.csv | 2147 rows x 101 columns

Majority-class baseline (3 classes): 70.3 of 100

Missing values left (to impute on the training set):
education_years            2
educ_spouse              126
occup_spouse               1
income                     3
family_member              1
ins_type                   4
ins_location               3
daughter_age_youngest      9
daughter_age_oldest        9

Parents per division:
meta_division
Dhaka         560
Chattogram    433
Rajshahi      265
Khulna        231
Rangpur       228
Mymensingh    160
Sylhet        143
Barishal      127
